In [0]:
# =============================================================
# search_utils
# Shared utility functions for vector search and retrieval
# =============================================================

In [0]:
def wait_for_index(index, max_retries=20):
    """
    Polls the vector index until it reaches ONLINE state or fails.
    
    Args:
        index       (str): The full index name
        max_retries (int): Maximum number of 30s polling attempts (default 20)
    """
    retries = 0
    while retries < max_retries:
        status = vsc.get_index(endpoint_name, index).describe().get("status", {})
        state = status.get("detailed_state", "UNKNOWN")

        if "ONLINE" in state:
            logger.info(f"🟢 Index '{index}' is ONLINE.")
            return
        elif "FAILED" in state:
            logger.error(f"❌ Index '{index}' reached a failure state: {state}.")
            return

        logger.info(f"⏳ Index '{index}' state: {state}... (Waiting 30s)")
        time.sleep(30)
        retries += 1
    else:
        raise TimeoutError(f"Index '{index}' timed out after {max_retries} retries.")


In [0]:
def search_documents(query, source_type, num_results=4):
    """
    Queries the vector index for a given source type and returns
    relevant chunks with citations.

    Args:
        query       (str): The search query
        source_type (str): The source domain to search e.g. 'books' or 'docs'
        num_results (int): Number of results to return (default 4)
    
    Returns:
        list: Raw data array of matching chunks
    """
    if source_type not in source_config:
        raise ValueError(f"Invalid source_type '{source_type}'. Must be one of: {list(source_config.keys())}")

    index = vsc.get_index(endpoint_name, source_config[source_type]["index"])

    results = index.similarity_search(
        query_text=query,
        columns=["content", "source", "page_number", "start_index"],
        num_results=num_results
    )

    return results.get('result', {}).get('data_array', [])

In [0]:
def run_diagnostic(source_type, query, num_results=4):
    """
    Interactive diagnostic tool for testing vector search results.

    Args:
        source_type (str): The source domain to search e.g. 'books' or 'docs'
        query       (str): The search query to test
        num_results (int): Number of results to return (default 4)
    """
    print(f"\n📡 RAW VECTOR SEARCH DIAGNOSTIC [{source_type.upper()}]: '{query}'")
    print("="*70)

    try:
        docs = search_documents(query, source_type, num_results)

        if not docs:
            print("⚠️ No matches found in the Vector Index.")
        else:
            for i, doc in enumerate(docs):
                content, source, page, start_idx = doc[0], doc[1], doc[2], doc[3]
                print(f"📍 [Match {i+1}] | File: {source} | Page: {page} | Index: {start_idx}")
                print(f"📄 \"{content[:300]}...\"")
                print("-" * 50)

    except Exception as e:
        logger.error(f"❌ Diagnostic failed: {e}")